# 01 Reproduce the base paper (Step 2)

**Base model (Lim et al. 2026, DM_v3_pooled).** For each country the annual total is known; monthly shares are `softmax(log historical shares + pooled month deviations + year effect)` and monthly counts follow a Dirichlet-multinomial.

**Cancellation.** The year effect is the same for all 12 months, so it cancels in the softmax: every held-out year of a country gets the same predicted shape. We check this numerically.

**What runs here** (our re-implementation in `dengue_recon.base_paper.lim_dm`):
- *MAP*: the same log posterior maximised with L-BFGS (about 5 s per fold after compilation) - run live;
- *NUTS*: NumPyro, 2 chains x (500 warm-up + 500 draws) - precomputed in `results/repro_lim/folds/`; set `RUN_NUTS = True` to refit (about 20 min per fold on 4 CPUs).

Reduced design vs the paper: 1 repetition (rep01) x interpolation mask x 3 folds (paper: 3 repetitions, 4 chains x 1,000 draws).

In [ ]:
# ---- Setup: works in Google Colab and locally (VS Code / Jupyter) ----
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/aminah9945/dengue-climate-reconstruction"
BRANCH = "main"          # use "official-run" if the pull request is not merged yet
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO = Path("/content/dengue-climate-reconstruction")
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[all]"], check=True)
    # Optional: keep the downloaded data on Google Drive so it survives the session.
    USE_DRIVE = False
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ.setdefault("DATA_DIR", "/content/drive/MyDrive/dengue_recon_data")
else:
    # Local: this notebook lives in <repo>/notebooks/
    REPO = Path(os.environ.get("DENGUE_ROOT", Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())).resolve()

os.environ.setdefault("DENGUE_ROOT", str(REPO))
os.environ.setdefault("DATA_DIR", str(REPO / "data"))
os.environ.setdefault("RESULTS_DIR", str(REPO / "results"))
os.environ.setdefault("XLA_FLAGS", "--xla_force_host_platform_device_count=1")
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))      # makes the package importable without a kernel restart

from dengue_recon import config as C
C.ensure_dirs()
print("repo       :", C.ROOT)
print("DATA_DIR   :", C.DATA_DIR, "| processed data present:", (C.DATA_DIR / "dengue_weekly.parquet").exists())
print("RESULTS_DIR:", C.RESULTS_DIR)

In [ ]:
import json
import numpy as np, pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option("display.width", 200, "display.max_columns", 30)


def show(path, width=900):
    p = Path(path)
    p = p if p.is_absolute() else C.FIG_DIR / p
    display(Image(str(p), width=width)) if p.exists() else print("missing figure:", p)


def have_data():
    ok = (C.DATA_DIR / "dengue_weekly.parquet").exists()
    if not ok:
        print("Processed data not found in DATA_DIR - run notebook 00 first (or set DATA_DIR).")
    return ok

In [ ]:
RUN_MAP = True
RUN_NUTS = False
from dengue_recon.base_paper import reproduce
from dengue_recon.base_paper.fetch import fetch_lim_mask
if RUN_MAP or RUN_NUTS:
    fetch_lim_mask()
    for f in (1, 2, 3):
        if RUN_MAP:
            reproduce.run("MAP", f, force=True)
        if RUN_NUTS:
            reproduce.run("NUTS", f)
tab = reproduce.table()

**Reading the table.** `MAE_median`/`RMSE_median` use the posterior-predictive median (the definition of the published table); `MAE`/`RMSE` use the mean. `COV80` is 80 % interval coverage, `CRPS` the probabilistic score. `JSD_mean` and `peak_month_err` are our shape metrics. `uniform` and `historical_mean_shares` are reference rows on the same test months.

In [ ]:
show("fig_repro_lim.png")

### Cancellation check (max |share difference| between held-out years of the same country)

In [ ]:
pd.read_csv(C.REPRO_DIR / "cancellation_check.csv").groupby("fit").max_abs_diff.describe()